<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-09/AI_Modul_9.9_Praktikum_Framework_Deep_Learning.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 9.9: Praktikum Framework Deep Learning (PyTorch)
**Mata Kuliah:** AI Academy  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Membangun model Multi-Layer Perceptron berorientasi objek menggunakan `torch.nn.Module`.
2. Mengonfigurasi pipeline pemuatan data efisien menggunakan `TensorDataset` dan `DataLoader`.
3. Mengimplementasikan siklus pelatihan kanonik (*canonical training loop*) dengan Autograd dan optimizer AdamW.
4. Mengevaluasi performa klasifikasi bibit sawit dan mengekspor model ke format standar ONNX.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Tersedia   : {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Perangkat Aktif : {device}')


## 1. Pembangkitan Data Varietas Bibit Sawit dan Pipa Pemuatan Data (DataLoader)
Membuat dataset 6 fitur morfometri bibit kelapa sawit (tinggi, diameter batang, helai daun, dll.) untuk 3 varietas: Dura, Pisifera, dan Tenera.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

n_samples = 600
n_features = 6

# Pembangkitan data sintetis terdistribusi normal
X_data = torch.randn(n_samples, n_features, dtype=torch.float32)
y_data = torch.randint(0, 3, (n_samples,), dtype=torch.long)

# Pemisahan Train (80%) dan Test (20%)
train_size = int(0.8 * n_samples)
X_train, X_test = X_data[:train_size], X_data[train_size:]
y_train, y_test = y_data[:train_size], y_data[train_size:]

# Pembungkusan DataLoader
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f'Jumlah Data Latih: {len(train_dataset)}, Data Uji: {len(test_dataset)}')


## 2. Perancangan Model PyTorch (AgriculturalMLP)
Membangun arsitektur jaringan saraf berorientasi objek dengan Batch Normalization, ReLU, dan Dropout.

In [ ]:
class AgriculturalMLP(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=32, num_classes=3, dropout_rate=0.2):
        super(AgriculturalMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )
        
    def forward(self, x):
        return self.net(x)

model = AgriculturalMLP(input_dim=6, hidden_dim=32, num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-4)
print(model)


## 3. Eksekusi Training Loop Kanonik dan Evaluasi
Melatih model selama 15 epoch dan mencatat dinamika loss serta akurasi.

In [ ]:
epochs = 15
history_loss = []
history_acc = []

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_x.size(0)
        _, preds = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (preds == batch_y).sum().item()
        
    epoch_loss = total_loss / total
    epoch_acc = (correct / total) * 100
    history_loss.append(epoch_loss)
    history_acc.append(epoch_acc)
    
print(f'Training Selesai! Final Train Loss: {history_loss[-1]:.4f} - Akurasi: {history_acc[-1]:.1f}%')

# Evaluasi pada Data Uji
model.eval()
test_correct = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        _, preds = torch.max(outputs, 1)
        test_correct += (preds == batch_y).sum().item()

test_acc = (test_correct / len(test_dataset)) * 100
print(f'Akurasi Evaluasi Test Set: {test_acc:.1f}%')


## 4. Visualisasi Kurva Pembelajaran dan Ekspor ke ONNX
Menampilkan grafik konvergensi pelatihan dan mengekspor model teruji ke format terbuka ONNX.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(history_loss, color='crimson', linewidth=2)
ax1.set_title('Dinamika Training Loss (Cross-Entropy)', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')

ax2.plot(history_acc, color='forestgreen', linewidth=2)
ax2.set_title('Peningkatan Akurasi Pelatihan (%)', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Akurasi (%)')

plt.tight_layout()
plt.savefig('praktikum_pytorch_training_9_9.png', dpi=150)
plt.show()

# Ekspor model ke ONNX
dummy_in = torch.randn(1, 6, device=device)
onnx_path = 'model_bibit_sawit.onnx'
torch.onnx.export(
    model,
    dummy_in,
    onnx_path,
    export_params=True,
    opset_version=14,
    input_names=['fitur_bibit'],
    output_names=['prediksi_varietas']
)
print(f'[OK] Model PyTorch berhasil diekspor ke: {onnx_path}')
